# Tutorial: Managing Google Cloud IAM with `iam.py`

This notebook demonstrates how to use the `Policy` class from the `iam.py` module to manage Identity and Access Management (IAM) policies and custom roles for your Google Cloud projects. The `iam.py` module simplifies interactions with Google Cloud's Resource Manager and IAM Admin APIs, providing an intuitive interface for common IAM operations.

## Setup: Install Libraries and Authenticate

In [ ]:
# Install necessary Google Cloud client libraries.
!pip install --upgrade google-cloud-resourcemanager google-cloud-iam-admin google-api-core

### Google Cloud Project and Authentication
To run this notebook, you'll need:
1.  **A Google Cloud Project ID**: Replace `YOUR_PROJECT_ID` below with your actual project ID.
2.  **Authentication**: Ensure your environment is authenticated to Google Cloud. The easiest way for local development or Colab is to use Application Default Credentials. Run the following command in your terminal or gcloud CLI (if running locally):
    ```bash
    gcloud auth application-default login
    ```
    In a Colab environment, you can authenticate using `google.colab.auth.authenticate_user()`:
    ```python
    from google.colab import auth
    auth.authenticate_user()
    ```
    Make sure the authenticated account has the necessary permissions for IAM operations on the specified project, such as `resourcemanager.projects.getIamPolicy`, `resourcemanager.projects.setIamPolicy`, `iam.roles.list`, and `iam.roles.create`.

In [ ]:
# Replace with your Google Cloud Project ID
PROJECT_ID = "YOUR_PROJECT_ID" 

if PROJECT_ID == "YOUR_PROJECT_ID":
    raise ValueError("Please replace 'YOUR_PROJECT_ID' with your actual Google Cloud Project ID.")

print(f"Using Google Cloud Project ID: {PROJECT_ID}")

## Embedding the `iam.py` Module

Since the `iam.py` module depends on `conidk.core.base` which is not provided, we will create dummy `Auth` and `Config` classes to allow the `Policy` class to initialize without errors. This allows us to embed the `iam.py` source code directly into this notebook for a self-contained demonstration.

In [ ]:
# --- Start of Mocking conidk.core.base --- 
import sys
from types import ModuleType

# Create dummy modules for the `conidk.core.base` hierarchy
sys.modules['conidk'] = ModuleType('conidk')
sys.modules['conidk.core'] = ModuleType('conidk.core')
sys.modules['conidk.core.base'] = ModuleType('conidk.core.base')

# Define dummy Auth and Config classes that will be assigned to the dummy base module
class DummyAuth:
    def __init__(self):
        pass # print("Dummy Auth handler initialized.")

class DummyConfig:
    def __init__(self):
        pass # print("Dummy Config handler initialized.")

# Assign the dummy classes to the dummy module
sys.modules['conidk.core.base'].Auth = DummyAuth
sys.modules['conidk.core.base'].Config = DummyConfig

base = sys.modules['conidk.core.base'] # Make `base` available for the `iam.py` code
# --- End of Mocking conidk.core.base --- 


# --- Start of iam.py code --- 
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""Manages IAM policies for Google Cloud projects.

This module provides the `Policy` class, which simplifies interactions with
Google Cloud IAM. It allows for retrieving, setting, and modifying IAM policies,
as well as managing custom roles within a specified Google Cloud project. The
class wraps the `google.cloud.resourcemanager_v3` and `google.cloud.iam_admin_v1`
client libraries to offer an intuitive interface for common IAM operations.
"""

# pylint: disable= no-member,

from typing import Any, Dict, List, Optional
from google.protobuf.json_format import ParseDict  # type: ignore
from google.protobuf.json_format import MessageToDict

from google.cloud import iam_admin_v1  # type: ignore
from google.cloud.iam_admin_v1 import types as iam_types  # types: ignore
from google.cloud import resourcemanager_v3
from google.cloud.iam_admin_v1.types import CreateRoleRequest
from google.cloud.iam_admin_v1.types import Role


from google.iam.v1 import policy_pb2  # type: ignore
# from conidk.core import base # This import is now handled by the mocking above


class Policy:
    """Manages IAM policies and custom roles for a Google Cloud project.

    This class provides a simplified interface for handling IAM policies and custom
    roles by wrapping the `resourcemanager_v3.ProjectsClient` and
    `iam_admin_v1.IAMClient`. It supports getting, setting, and modifying IAM
    policies, as well as listing and creating custom roles for a specific project.

    Attributes:
        project_id (str): The Google Cloud project ID.
        project_name (str): The formatted project name required by the APIs
            (e.g., "projects/your-project-id").
        auth (base.Auth): An authentication handler object.
        config (base.Config): A configuration handler object.
        client (resourcemanager_v3.ProjectsClient): The client for interacting
            with the Cloud Resource Manager API.
        iam_client (iam_admin_v1.IAMClient): The client for interacting with the
            IAM Admin API.
    """

    def __init__(
        self,
        project_id: str,
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    ) -> None:
        """Initializes the Policy class.

        Args:
            project_id: The unique identifier for the Google Cloud project.
            auth: Optional. An authentication object.
            config: Optional. A configuration object.
        """

        self.auth = auth or base.Auth()
        self.config = config or base.Config()

        self.project_id = project_id
        self.project_name = f"projects/{project_id}"
        self.client = resourcemanager_v3.ProjectsClient()
        self.iam_client = iam_admin_v1.IAMClient()

    def set(self, policy: Dict[str, Any]) -> policy_pb2.Policy:
        """Sets the IAM policy for the project.

        Args:
            policy: A dictionary representing the IAM policy.

        Returns:
            The updated policy_pb2.Policy object.

        Raises:
            google.api_core.exceptions.GoogleAPICallError: If the API call fails.
        """

        # The `set_iam_policy` method accepts a dictionary that mirrors
        # the `SetIamPolicyRequest` structure.
        request = {"resource": f"projects/{self.project_id}", "policy": policy}

        return self.client.set_iam_policy(request=request)

    def get(self) -> policy_pb2.Policy:
        """Gets the current IAM policy for the project.

        Returns:
            The current policy_pb2.Policy object.

        Raises:
            google.api_core.exceptions.GoogleAPICallError: If the API call fails.
        """
        request = {
            "resource": f"projects/{self.project_id}",
        }
        return self.client.get_iam_policy(request=request)

    def add(self, member: str, role: str) -> None:
        """Adds a new member and role binding to the project's IAM policy.

        Args:
            member: The identifier of the member to add.
            role: The IAM role to assign to the member.
        """

        policy = self.client.get_iam_policy(resource=self.project_name)
        policy_dict = MessageToDict(policy)

        # Find the binding for the given role, or create a new one
        binding_found = False
        for binding in policy_dict.get("bindings", []):
            if binding.get("role") == role:
                if member not in binding.get("members", []):
                    binding.get("members", []).append(member)
                binding_found = True
                break

        if not binding_found:
            policy_dict.setdefault("bindings", []).append(
                {"role": role, "members": [member]}
            )

        policy_object = ParseDict(policy_dict, policy_pb2.Policy())
        self.client.set_iam_policy(
            request={"resource": self.project_name, "policy": policy_object}
        )

    def list_custom_roles(self) -> Optional[list[str]]:
        """Lists the names of custom IAM roles within the project.

        Returns:
            A list of custom role names or None if no custom roles are found.
        """
        request = iam_admin_v1.ListRolesRequest(
            parent=self.project_name,
            view=iam_types.RoleView.FULL,
            show_deleted=False,  # Set to True to include deleted roles
        )

        page_result = self.iam_client.list_roles(request=request)
        roles = []
        custom_roles_found = False
        for role in page_result:
            # Custom roles have a path that includes "organizations/" or "projects/"
            if "organizations/" in role.name or "projects/" in role.name:
                roles.append(role.name)
                custom_roles_found = True

        if not custom_roles_found:
            return None
        return roles

    def create_custom_role(
        self,
        role_id: str,
        permissions: List[str],
        title: Optional[str] = None,
        description: Optional[str] = None,
    ) -> Role:
        """Creates a new custom IAM role within the project.

        Args:
            role_id: The unique identifier for the custom role.
            permissions: A list of IAM permissions to include in the role.
            title: An optional display title for the role.
            description: An optional description for the role.

        Returns:
            The created `iam_admin_v1.types.Role` object.
        """

        # Create the role object with the desired permissions and title
        role_definition = Role(
            title=title if title else role_id,  # Use role_id as title if not provided
            included_permissions=permissions,
            stage=Role.RoleLaunchStage.BETA,
            description=description,
        )

        request = CreateRoleRequest(
            parent=self.project_name, role_id=role_id, role=role_definition
        )

        return self.iam_client.create_role(request=request)

# --- End of iam.py code ---

## Initialize the `Policy` Manager

In [ ]:
# Instantiate the Policy class
iam_policy_manager = Policy(project_id=PROJECT_ID)
print(f"Policy manager initialized for project: {iam_policy_manager.project_id}")

## Working with Project IAM Policies

The `Policy` class allows you to retrieve, add members to roles, and set the complete IAM policy for your project.

### 1. Get Current IAM Policy

Use the `get()` method to retrieve the current IAM policy for the project. The output is a `policy_pb2.Policy` object, which can be easily converted to a dictionary for readability.

In [ ]:
from google.protobuf.json_format import MessageToDict

try:
    current_policy = iam_policy_manager.get()
    print("Current IAM Policy:")
    print(MessageToDict(current_policy))
except Exception as e:
    print(f"Error getting IAM policy: {e}")
    print("Please ensure your authenticated account has 'resourcemanager.projects.getIamPolicy' permission.")

### 2. Add Member to a Role

The `add(member, role)` method simplifies adding a new member to an existing or new role binding within the project's IAM policy. This method internally fetches the current policy, modifies it, and then sets it. 

**Important**: Replace `MEMBER_EMAIL` with a valid email (e.g., a user account like `user:your-email@example.com` or a service account like `serviceAccount:your-service-account@your-project-id.iam.gserviceaccount.com`). Also, choose an appropriate `ROLE` (e.g., `roles/viewer`, `roles/editor`).

In [ ]:
# Define the member and role to add
MEMBER_TO_ADD = "user:MEMBER_EMAIL"  # e.g., "user:test-user@example.com"
ROLE_TO_GRANT = "roles/viewer"

if MEMBER_TO_ADD == "user:MEMBER_EMAIL":
    print("Skipping add member example: Please replace 'MEMBER_EMAIL' with a valid member identifier (user or service account email).")
else:
    print(f"Adding {MEMBER_TO_ADD} to role {ROLE_TO_GRANT}...")
    try:
        iam_policy_manager.add(member=MEMBER_TO_ADD, role=ROLE_TO_GRANT)
        print(f"Successfully added {MEMBER_TO_ADD} to {ROLE_TO_GRANT}.")
    except Exception as e:
        print(f"Error adding member: {e}")
        print("Please ensure your authenticated account has 'resourcemanager.projects.setIamPolicy' permission.")
        print("Also, verify that the member and role formats are correct.")

### 3. Verify Changes (Get Policy Again)

Let's retrieve the policy again to confirm that the member and role binding have been added.

In [ ]:
if MEMBER_TO_ADD != "user:MEMBER_EMAIL":
    print("Updated IAM Policy after adding member:")
    try:
        updated_policy = iam_policy_manager.get()
        print(MessageToDict(updated_policy))
    except Exception as e:
        print(f"Error getting updated IAM policy: {e}")

### 4. Set Complete IAM Policy

The `set(policy_dict)` method allows you to completely replace the project's IAM policy with a new one provided as a dictionary. This is useful for more complex modifications, such as removing members, modifying conditions, or performing bulk updates. 

It's crucial to always fetch the current policy first, modify its dictionary representation (including the `etag` for optimistic locking), and then set the modified policy. Otherwise, you risk overwriting concurrent changes.

In [ ]:
from google.protobuf.json_format import ParseDict
from google.iam.v1 import policy_pb2

print("Demonstrating `set()` method: removing the previously added member (if applicable).")

try:
    # 1. Get the current policy
    policy_to_modify = iam_policy_manager.get()
    policy_dict = MessageToDict(policy_to_modify)

    # 2. Modify the policy dictionary (e.g., remove the member added previously)
    # Iterate through bindings and remove the member if found
    if 'bindings' in policy_dict:
        for binding in policy_dict['bindings']:
            if binding.get('role') == ROLE_TO_GRANT and 'members' in binding:
                if MEMBER_TO_ADD in binding['members']:
                    binding['members'].remove(MEMBER_TO_ADD)
                    print(f"Removed {MEMBER_TO_ADD} from {ROLE_TO_GRANT} binding in policy dictionary.")
                # Optionally remove binding if no members are left, or it might be ignored by the API
                # binding['members'] = [m for m in binding['members'] if m != MEMBER_TO_ADD]

        # Clean up empty bindings if desired (optional, API might handle it)
        policy_dict['bindings'] = [b for b in policy_dict['bindings'] if b.get('members')]

    # 3. Set the modified policy
    print("Attempting to set modified policy...")
    set_result = iam_policy_manager.set(policy_dict)
    print("Policy successfully set. New policy (truncated):")
    print(MessageToDict(set_result)['bindings'][:2]) # Print first two bindings for brevity

except Exception as e:
    print(f"Error demonstrating `set()` method: {e}")
    print("Please ensure your authenticated account has 'resourcemanager.projects.setIamPolicy' permission.")

## Managing Custom IAM Roles

The `Policy` class also provides methods to list and create custom IAM roles within your project. Custom roles allow you to define a granular set of permissions that are specific to your needs.

### 1. List Custom Roles

Use `list_custom_roles()` to see all custom roles defined for your project. This only lists roles explicitly created within your project scope (or organization if parent was organization).

In [ ]:
print("Listing custom roles for the project...")
try:
    custom_roles = iam_policy_manager.list_custom_roles()
    if custom_roles:
        print("Found custom roles:")
        for role_name in custom_roles:
            print(f"- {role_name}")
    else:
        print("No custom roles found for this project.")
except Exception as e:
    print(f"Error listing custom roles: {e}")
    print("Please ensure your authenticated account has 'iam.roles.list' permission.")

### 2. Create a Custom Role

The `create_custom_role()` method allows you to define and create a new custom role with a specific set of permissions. 

**Note**: Custom role IDs must be unique within the project and follow a specific naming convention (lowercase letters, numbers, hyphens). Permissions must be valid IAM permissions (e.g., `compute.instances.get`, `storage.objects.list`).

In [ ]:
NEW_ROLE_ID = "myCustomViewerRole" # Must be unique and follow naming conventions
NEW_ROLE_TITLE = "My Custom Viewer Role"
NEW_ROLE_DESCRIPTION = "A custom role with basic view permissions for compute and storage."
NEW_ROLE_PERMISSIONS = [
    "compute.instances.get",
    "compute.instances.list",
    "storage.objects.get",
    "storage.objects.list"
]

print(f"Attempting to create custom role: {NEW_ROLE_ID}...")
try:
    created_role = iam_policy_manager.create_custom_role(
        role_id=NEW_ROLE_ID,
        permissions=NEW_ROLE_PERMISSIONS,
        title=NEW_ROLE_TITLE,
        description=NEW_ROLE_DESCRIPTION
    )
    print(f"Custom role '{created_role.name}' created successfully.")
    print(f"Title: {created_role.title}")
    print(f"Permissions: {created_role.included_permissions}")
except Exception as e:
    print(f"Error creating custom role: {e}")
    print("This might be due to existing role ID, insufficient permissions ('iam.roles.create'), or invalid permissions in the list.")

### 3. Verify Custom Role Creation

Let's list the custom roles again to ensure our new role appears in the list.

In [ ]:
print("Listing custom roles again to verify creation...")
try:
    custom_roles_after_creation = iam_policy_manager.list_custom_roles()
    if custom_roles_after_creation:
        print("Found custom roles:")
        for role_name in custom_roles_after_creation:
            print(f"- {role_name}")
    else:
        print("No custom roles found.")
except Exception as e:
    print(f"Error listing custom roles: {e}")

## Conclusion

This tutorial demonstrated how the `Policy` class from `iam.py` simplifies common IAM operations in Google Cloud, including retrieving and setting project policies, adding members to roles, and managing custom roles. By wrapping the underlying Google Cloud client libraries, it provides a more streamlined and developer-friendly interface for IAM management tasks.

## Cleanup (Optional but Recommended)

If you created a custom role during this tutorial, you might want to delete it to avoid cluttering your project or incurring unnecessary costs (though custom roles themselves don't directly cost money, they do count towards quotas).

The `iam.py` module does not expose a `delete_custom_role` method directly, but you can use the underlying `iam_client` to perform this action. Ensure your authenticated account has the `iam.roles.delete` permission.

```python
# Example of how to delete a custom role directly using the IAM client
# from google.cloud import iam_admin_v1

# iam_client = iam_admin_v1.IAMClient()
# role_name_to_delete = f"projects/{PROJECT_ID}/roles/{NEW_ROLE_ID}" # e.g., 'projects/YOUR_PROJECT_ID/roles/myCustomViewerRole'

# try:
#     iam_client.delete_role(name=role_name_to_delete)
#     print(f"Successfully deleted custom role: {role_name_to_delete}")
# except Exception as e:
#     print(f"Error deleting custom role {role_name_to_delete}: {e}")
```